In [ ]:
!pip install langchain==0.3.24 -q
!pip install langchain-openai -q
!pip install langchain-community -q
!pip install langgraph -q
!pip install -U mypy_extensions -q
!pip install langchain-classic -q

!pip install langchain-tavily -q
!pip install pyowm -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 72.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.12 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.5 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('API_KEY')

In [ ]:
llm = ChatOpenAI(model = "gpt-4o")

In [ ]:
from langchain_tavily import TavilySearch
from langchain_community.utilities import OpenWeatherMapAPIWrapper
from langchain_classic.agents import load_tools

os.environ["OPENWEATHERMAP_API_KEY"] = userdata.get('OPENWEATHERMAP_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

# Initialize google search tool
search_tool = TavilySearch()

# Initialize OpenWeatherMap tool
weather = OpenWeatherMapAPIWrapper()

weather_tool = load_tools(["openweathermap-api"], llm)[0]

/tmp/ipykernel_1748/3877581714.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import OpenWeatherMapAPIWrapper


In [ ]:
tools = [search_tool, weather_tool]

In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def llm_call(state:AgentState) -> AgentState:
    system_prompt = SystemMessage(content=
        "You are an intelligent AI assistant."
    )
    response = llm_with_tools.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


def decision(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("agent", llm_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")
graph.add_conditional_edges(
    "agent",
    decision,
    {
        "continue": "tools",
        "end": END,
    },
)
graph.add_edge("tools", "agent")

app = graph.compile()

Step by step visibility

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is the current weather in Kandy?")]
})

result['messages'][-1].content

"The current weather in Kandy, Sri Lanka, is experiencing moderate rain. Here's the detailed weather information:\n\n- Temperature: 26.98°C (feels like 29.51°C)\n- Rainfall: 1.84 mm in the last hour\n- Humidity: 78%\n- Wind: 0.95 m/s, coming from 1°\n- Cloud cover: 100%"

In [ ]:
events = app.stream({
    "messages": [HumanMessage(content="What is the current weather in Kandy?")]
}, stream_mode="values")

for event in events:
    print(event)

{'messages': [HumanMessage(content='What is the current weather in Kandy?', additional_kwargs={}, response_metadata={}, id='6fc3a555-f75a-4255-a307-ccd542b37d3a')]}
{'messages': [HumanMessage(content='What is the current weather in Kandy?', additional_kwargs={}, response_metadata={}, id='6fc3a555-f75a-4255-a307-ccd542b37d3a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 1251, 'total_tokens': 1267, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_0a4b2d353e', 'id': 'chatcmpl-EXKuJuNFsp43jmtDWYslVQe5lt9FL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12482-92b5-7c81-8e14-572f2ed8b9

In [ ]:
events = app.stream({
    "messages": [HumanMessage(content="What is the current weather in Kandy?")]
}, stream_mode="updates")

for event in events:
    print(event)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 1251, 'total_tokens': 1270, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_0a4b2d353e', 'id': 'chatcmpl-EXKvsPDnZmK2kjCyc8zEahmSKSGwt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12484-06e9-7ed0-834e-cfc0f47897a7-0', tool_calls=[{'name': 'open_weather_map', 'args': {'location': 'Kandy, Sri Lanka'}, 'id': 'call_uA3aiiVsuxR7OYr91XnxUDK1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1251, 'output_tokens': 19, 'total_tokens': 1270, 'input_token_details': {'audio': 0, 'cache_read':

Real-time response

In [ ]:
events = app.astream_events({
    "messages": [HumanMessage(content="What is the current weather in Kandy?")]
}, version='v2')

async for event in events:
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='What is the current weather in Kandy?', additional_kwargs={}, response_metadata={})]}}, 'name': 'LangGraph', 'tags': [], 'run_id': '01a12486-0ad5-7812-802a-4964a112ac54', 'metadata': {'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='What is the current weather in Kandy?', additional_kwargs={}, response_metadata={}, id='9e11189d-6871-4686-9b13-47ff5f83cc71')]}}, 'name': 'agent', 'tags': ['graph:step:1'], 'run_id': '01a12486-0ad7-7381-9025-874822c6e70c', 'metadata': {'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'agent', 'langgraph_triggers': ('branch:to:agent',), 'langgraph_path': ('__pregel_pull', 'agent'), 'langgraph_checkpoint_ns': 'agent:32732388-46fd-7b25-426c-fd4bd0213269'}, 'parent_ids': ['01a12486-0ad5-7812-802a-4964a112ac54']}
{'event': 'on_chat_model_start', 'data': {'input': {'messages': [[

In [ ]:
events = app.astream_events({
    "messages": [HumanMessage(content="What is the current weather in Kandy?")]
}, version='v2')

async for event in events:
    if event['event'] == "on_chat_model_stream":
      print(event['data']['chunk'].content, end='', flush=True)

The current weather in Kandy is characterized by moderate rain. Here are the details:

- Temperature: 26.98°C (feels like 29.51°C)
- Humidity: 78%
- Wind: 0.95 m/s from 1°
- Rain: 1.84 mm in the last hour
- Cloud Cover: 100%

Stay dry if you're heading out!